> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 4 – Gráficos de controle de qualidade

**Objetivo:** dominar `fig, axes = plt.subplots(...)` e usar gráficos para encontrar problemas nos dados.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import carregar_dominio, preencher_nan, preparar_pastas, FIG_DIR, NIR_FILE, MIR_FILE
preparar_pastas()

In [ ]:
meta_nir, X_nir, wl_nir = carregar_dominio(NIR_FILE)
meta_mir, X_mir, wl_mir = carregar_dominio(MIR_FILE)
X_nir, X_mir = preencher_nan(X_nir), preencher_nan(X_mir)

## Exercício 1 – Função `plot_qc`
Painel esquerdo: até 200 espectros com `alpha=0.05`. Painel direito: média ± 1 desvio padrão com `fill_between`. Inverta o eixo x quando `inverter=True`.

In [ ]:
def plot_qc(wl, X, titulo, inverter=False, unidade='nm'):
    m, s = X.mean(0), X.std(0)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(wl, X[:200].T, alpha=0.05, color='tab:blue')
    ax[0].set_title(f'{titulo}: espectros brutos')
    ax[1].plot(wl, m, color='tab:blue')
    ax[1].fill_between(wl, m - s, m + s, alpha=0.2)
    ax[1].set_title(f'{titulo}: média ± 1 DP')
    for a in ax:
        a.set_xlabel(unidade)
        a.set_ylabel('Reflectância')
        a.grid(alpha=0.2)
        if inverter:
            a.invert_xaxis()
    fig.tight_layout()
    return fig

## Exercício 2 – Gerar e salvar as figuras
Lembre: `savefig` **antes** de `plt.show()`.

In [ ]:
fig = plot_qc(wl_nir, X_nir, 'NIR', unidade='nm')
fig.savefig(FIG_DIR / '01_qc_nir.png', dpi=150)
plt.show()

fig = plot_qc(wl_mir, X_mir, 'MIR', inverter=True, unidade='cm-1')
fig.savefig(FIG_DIR / '01_qc_mir.png', dpi=150)
plt.show()

## Exercício 3 – Marcar regiões de interesse
No gráfico da média do NIR, desenhe linhas verticais (`axvline`) em 1000 e 1800 nm (emendas dos detectores) e em 1400, 1900 e 2200 nm (absorções de água e argila).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wl_nir, X_nir.mean(0))
for x in (1000, 1800):
    ax.axvline(x, color='tab:green', ls='--', label='Emenda de detector' if x == 1000 else None)
for x in (1400, 1900, 2200):
    ax.axvline(x, color='purple', ls=':', label='Absorção (água/argila)' if x == 1400 else None)
ax.set_xlabel('nm'); ax.legend(); ax.grid(alpha=0.2)
plt.show()

## Checkpoint

Aponte no gráfico do NIR as regiões com mais ruído. Há "degraus" perto de 1000 nm e 1800 nm?